In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
import re

# === Parametri da modificare ===
dataset_dir = "C:\\Users\\geard\\OneDrive\\Download\\Tesi\\dati_ts\\dataset_2"        # <--- cartella dove hai i file del primo dataset
output_dir =  "C:\\Users\\geard\\OneDrive\\Download\\Tesi\\images_ts" # <--- dove salvare i plot
os.makedirs(output_dir, exist_ok=True)

# Carica i file .TXT
file_list = sorted(glob.glob(os.path.join(dataset_dir, "*.TXT")))

# Gruppi per le 3 serie: usiamo regex per separare per "#SetN#"
series_dict = {1: [], 2: [], 3: []}

for file_path in file_list:
    match = re.search(r"#Set(\d+)#", file_path)
    if match:
        serie_num = int(match.group(1))
        if serie_num in series_dict:
            series_dict[serie_num].append(file_path)

# Colori
colors = plt.cm.viridis(np.linspace(0, 1, 16))

# Funzione per caricare
def load_file(file_path):
    return pd.read_csv(file_path, sep="\t", skiprows=1)

# Plot per ogni serie
for serie_num, files in series_dict.items():
    if len(files) != 16:
        print(f"⚠️ Serie {serie_num} ha {len(files)} file invece di 16.")
        continue

    plt.figure(figsize=(10, 6))
    for j, file_path in enumerate(sorted(files)):
        df = load_file(file_path)
        plt.plot(df["Fn (mN)"], df["Pd (nm)"], label=f"Test {j+1}", color=colors[j])

    plt.title(f"Serie {serie_num} - {dataset_dir}")
    plt.xlabel("Fn (mN) - Forza applicata")
    plt.ylabel("Pd (nm) - Profondità")
    plt.grid(True)
    plt.legend(fontsize="small", loc="upper left")
    plt.tight_layout()

    # Salva
    filename = f"serie_{serie_num}.png"
    plt.savefig(os.path.join(output_dir, filename), dpi=300)
    plt.close()

print(f"✅ Grafici salvati in '{output_dir}'")


✅ Grafici salvati in 'C:\Users\geard\OneDrive\Download\Tesi\images_ts'


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import re

# === CONFIG ===
dataset1_dir = "C:\\Users\\geard\\OneDrive\\Download\\Tesi\\dati_ts\\dataset_1"
dataset2_dir = "C:\\Users\\geard\\OneDrive\\Download\\Tesi\\dati_ts\\dataset_2"
output_dir = "C:\\Users\\geard\\OneDrive\\Download\\Tesi\\images_ts"
os.makedirs(output_dir, exist_ok=True)

# === FUNZIONI ===

def load_files_by_series(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.TXT")))
    series_dict = {1: [], 2: [], 3: []}
    for file in files:
        match = re.search(r"#Set(\d+)#", file)
        if match:
            serie_num = int(match.group(1))
            if serie_num in series_dict:
                series_dict[serie_num].append(file)
    return series_dict

def load_file(file_path):
    return pd.read_csv(file_path, sep="\t", skiprows=1)

def commento_serie(delta_pd, fn_common):
    delta_mean = np.nanmean(delta_pd)
    max_diff = np.nanmax(delta_pd)
    min_diff = np.nanmin(delta_pd)
    
    if np.allclose(delta_mean, 0, atol=0.5):
        return "🟰 Nessuna differenza significativa tra i dataset."
    elif delta_mean > 0:
        return f"🔶 Il dataset 2 mostra una penetrazione mediamente maggiore (+{delta_mean:.2f} nm), con un massimo di +{max_diff:.2f} nm."
    else:
        return f"🔷 Il dataset 1 mostra una penetrazione mediamente maggiore ({delta_mean:.2f} nm), con un minimo di {min_diff:.2f} nm."

# === ESECUZIONE ===
dataset1_series = load_files_by_series(dataset1_dir)
dataset2_series = load_files_by_series(dataset2_dir)

for serie_num in [1, 2, 3]:
    files1 = sorted(dataset1_series[serie_num])
    files2 = sorted(dataset2_series[serie_num])
    
    if len(files1) != 16 or len(files2) != 16:
        print(f"⚠️ Serie {serie_num}: numero errato di file.")
        continue

    fn_common = np.linspace(0, 10, 500)  # griglia comune

    pd_interp1, pd_interp2 = [], []

    for f1, f2 in zip(files1, files2):
        df1 = load_file(f1)
        df2 = load_file(f2)

        pd1 = np.interp(fn_common, df1["Fn (mN)"], df1["Pd (nm)"], left=np.nan, right=np.nan)
        pd2 = np.interp(fn_common, df2["Fn (mN)"], df2["Pd (nm)"], left=np.nan, right=np.nan)

        pd_interp1.append(pd1)
        pd_interp2.append(pd2)

    pd_interp1 = np.array(pd_interp1)
    pd_interp2 = np.array(pd_interp2)

    mean_pd1 = np.nanmean(pd_interp1, axis=0)
    mean_pd2 = np.nanmean(pd_interp2, axis=0)
    std_pd1 = np.nanstd(pd_interp1, axis=0)
    std_pd2 = np.nanstd(pd_interp2, axis=0)

    delta_pd = mean_pd2 - mean_pd1
    delta_std = np.sqrt(std_pd1**2 + std_pd2**2)

    # === PLOT ===
    plt.figure(figsize=(10, 6))
    plt.plot(fn_common, delta_pd, label="ΔPd = Pd₂ - Pd₁", color="darkorange")
    plt.axhline(0, color='gray', linestyle='--')
    plt.title(f"Differenza Pd - Serie {serie_num}")
    plt.xlabel("Fn (mN)")
    plt.ylabel("ΔPd (nm)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"differenza_serie{serie_num}.png"), dpi=300)
    plt.close()

    # === CSV Export ===
    df_export = pd.DataFrame({
        "Fn (mN)": fn_common,
        "mean_Pd_dataset2": mean_pd2,
        "mean_Pd_dataset1": mean_pd1,
        "ΔPd (nm)": delta_pd,
        "±std (nm)": delta_std
    })
    df_export.to_csv(os.path.join(output_dir, f"dati_serie{serie_num}.csv"), index=False)

    # === COMMENTO automatico ===
    commento = commento_serie(delta_pd, fn_common)
    with open(os.path.join(output_dir, f"commento_serie{serie_num}.txt"), "w", encoding="utf-8") as f:
        f.write(commento)

    print(f"✅ Serie {serie_num}: grafico, dati e commento generati.")


C:\Users\geard\AppData\Local\Temp\ipykernel_31020\702579271.py:71: RuntimeWarning: Mean of empty slice
  mean_pd1 = np.nanmean(pd_interp1, axis=0)
C:\Users\geard\AppData\Local\Temp\ipykernel_31020\702579271.py:72: RuntimeWarning: Mean of empty slice
  mean_pd2 = np.nanmean(pd_interp2, axis=0)
C:\Users\geard\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


✅ Serie 1: grafico, dati e commento generati.


C:\Users\geard\AppData\Local\Temp\ipykernel_31020\702579271.py:71: RuntimeWarning: Mean of empty slice
  mean_pd1 = np.nanmean(pd_interp1, axis=0)
C:\Users\geard\AppData\Local\Temp\ipykernel_31020\702579271.py:72: RuntimeWarning: Mean of empty slice
  mean_pd2 = np.nanmean(pd_interp2, axis=0)
C:\Users\geard\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


✅ Serie 2: grafico, dati e commento generati.


C:\Users\geard\AppData\Local\Temp\ipykernel_31020\702579271.py:71: RuntimeWarning: Mean of empty slice
  mean_pd1 = np.nanmean(pd_interp1, axis=0)
C:\Users\geard\AppData\Local\Temp\ipykernel_31020\702579271.py:72: RuntimeWarning: Mean of empty slice
  mean_pd2 = np.nanmean(pd_interp2, axis=0)
C:\Users\geard\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


✅ Serie 3: grafico, dati e commento generati.


In [2]:
from PIL import Image
import os

for filename in os.listdir(r"C:\Users\geard\OneDrive\Download\Tesi\aa"):
    if filename.endswith(".png"):
        img = Image.open(os.path.join(r"C:\Users\geard\OneDrive\Download\Tesi\images_report", filename)).convert("RGB")
        new_name = filename.replace(".png", ".jpg")
        img.save(os.path.join(r"C:\Users\geard\OneDrive\Download\Tesi\images_report", new_name), quality=90)
